<a href="https://colab.research.google.com/github/simply-logical/mlbook_ii_notebooks/blob/master/Europarl_simplex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Europarl Simplex**

In this notebook we explore the decision boundary for different models and their configurations.

NOTE: Before running the cells, import the simplex.py file from [here](https://github.com/simply-logical/mlbook_ii_notebooks/blob/master/simplex.py). For that, click on the folder icon on the left, right button click, select "New File", paste the content of the file and save the file.  

USAGE: Run all the cells till reach the ones that generate the GUI output and select the appropriate alternatives.


In [ ]:
! pip install datasets pyarrow mlcroissant --quiet

In [ ]:
import nltk
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from absl import logging as absl_logging
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB


from simplex import SimplexColormap, plot_decision_regions, simplex_legend
from europarl_utils import EuroparlDataset, EuroparlDataCleaner, EuroparlDataProcessor, load_data, process_datasets, train_classifier

In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords')

In [ ]:
warnings.filterwarnings("ignore")
absl_logging.set_verbosity(absl_logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)

**Dataset loader**

As in the previous notebooks, run this cell to load the selected combination of languages.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

context = {}

dropdown = widgets.Dropdown(
    options=["Europarl"],
    value="Europarl",
    description='Dataset:',
    layout=widgets.Layout(width='auto')
)

lang_select = widgets.SelectMultiple(
    options=['en','pt', 'es', 'fr', 'da', 'de', 'fi','it', 'el', 'nl'],
    value=['pt'],
    description='Languages:',
    rows=5,
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

lang_help = widgets.HTML(
    value="<small>Hold <b>Ctrl</b> (Win) or <b>Cmd</b> (Mac) to select 3 items</small>",
    layout=widgets.Layout(margin='-10px 0 0 85px')
)

button = widgets.Button(
    description="Plot",
    button_style='info',
    layout=widgets.Layout(width='auto', height='40px')
)

output = widgets.Output()


def on_click_action(b):
    with output:
        clear_output()
        selected = lang_select.value

        if len(selected) != 3:
            print(f"Error: Select exactly 3 languages. (You selected {len(selected)})")
            return

        print(f"Generating dataset for Europarl with: {', '.join(selected)}")

        try:
            datasets = load_data(languages=selected)
            context['dataset'] = process_datasets(datasets)
            context['languages'] = selected
        except Exception as ex:
            print(f"Error:{ex}")

button.on_click(on_click_action)

grid = widgets.GridspecLayout(4, 1, grid_gap='10px')
grid[0, 0] = dropdown
grid[1, 0] = lang_select
grid[2, 0] = lang_help
grid[3, 0] = button

grid.layout.width = '400px'
grid.layout.margin = '20px'

display(grid, output)

**Model Selection and Configuration**

Run the cell below to load the GUI where you'll see the options for models and their configurations. After selecting the model and the configurations, click Run to train the model. Finally run the next cells for the plot

In [ ]:
models = {
    'Decision tree': DecisionTreeClassifier,
    'Naive Bayes': GaussianNB,
    'KNN': KNeighborsClassifier
}

parameters = {
    'Decision tree': {
        'criterion': ['gini', 'entropy'],
        'max_depth': [None, 1, 2, 3, 5, 10, 100]},
    'Naive Bayes': {},
    'KNN': {
        'n_neighbors': [1, 3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance']
    }
}

model_context = {}

model_dropdown = widgets.Dropdown(
    options=list(models.keys()),
    value=list(models.keys())[0],
    description='Model:',
    layout=widgets.Layout(width='auto')
)

params_container = widgets.VBox(layout=widgets.Layout(margin='10px 0'))

button = widgets.Button(
    description="Run",
    button_style='success',
    layout=widgets.Layout(width='auto', height='40px')
)

output = widgets.Output()

active_param_widgets = {}

def update_params_inputs(change):
    selected_model = change['new']
    model_params = parameters[selected_model]


    active_param_widgets.clear()
    children = []


    for param_name, param_options in model_params.items():
        options_clean = [str(opt) if opt is None else opt for opt in param_options]

        dropdown_param = widgets.Dropdown(
            options=options_clean,
            value=options_clean[0],
            description=f'{param_name}:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='auto')
        )
        active_param_widgets[param_name] = dropdown_param
        children.append(dropdown_param)

    if not children:
        children.append(widgets.HTML("<small>No configurable parameters.</small>"))

    params_container.children = children

model_dropdown.observe(update_params_inputs, names='value')
update_params_inputs({'new': model_dropdown.value})

def on_click_configure(b):
    with output:
        clear_output()
        selected_model_name = model_dropdown.value

        chosen_kwargs = {}
        for param_name, widget in active_param_widgets.items():
            val = widget.value
            if val == 'None':
                val = None
            chosen_kwargs[param_name] = val

        try:
            model_class = models[selected_model_name]
            configured_model = model_class(**chosen_kwargs)

            model_context['chosen_model'] = configured_model
            model_context['trained_model'] = train_classifier(context['dataset'], configured_model)

            print(f"Model instantiated")
            print(configured_model)
        except Exception as e:
            print(f"Error configuring the model: {e}")

button.on_click(on_click_configure)

grid = widgets.GridspecLayout(4, 1, grid_gap='10px')
grid[0, 0] = model_dropdown
grid[1, 0] = params_container
grid[2, 0] = button

grid.layout.width = '400px'
grid.layout.margin = '20px'

display(grid, output)

In [ ]:
clf = model_context['trained_model']
dataset = context['dataset']

In [ ]:
cmap = SimplexColormap(sharpen=3.0)
pad = 1.0
xlim = (0, 30)
ylim = (0, 50)

In [ ]:
fig, (ax, axL) = plt.subplots(1, 2, figsize=(11, 5),
                              gridspec_kw={"width_ratios":[3,1]})

plot_decision_regions(clf, xlim, ylim, ax=ax, cmap=cmap, n=300)

# overlay the training points using the same colour map at full confidence
onehot = np.eye(3)[dataset['label']]
ax.scatter(dataset['num_lowercase_words'].values,dataset['num_uppercase_words'].values, c=cmap(onehot), edgecolors="k", linewidths=0.4, s=22)

ax.set_title("Decision regions coloured by predicted class probabilities")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")

simplex_legend(ax=axL, cmap=cmap)
axL.set_title("colour key", fontsize=10)

plt.tight_layout()
plt.show()